# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/row-huh/ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Scoring / ranking**, not classification and not clustering.

Not classification: the review team doesn't need a binary yes/no flag, they need an ordered queue, because capacity is limited (~50 reviews/week against a ~9,759-page candidate pool from w01 — collapsing that to a flag throws away exactly the information that decides who gets looked at first). Not clustering: I already have a defined thing to predict (a CTR opportunity gap), I'm not exploring unlabeled structure. This also matches how the guide itself frames this lane: "Ranked opportunity score; position/volume adjusted."

In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
qualified = df[(df.impressions_90d > 0) & (df.content_age_days >= 90)].copy()

# confirm this is a continuous-score problem, not a clean binary split
print("unique ctr values:", qualified.ctr.nunique(), "/", len(qualified))
print(qualified.ctr.describe())


unique ctr values: 401 / 30000
count    30000.000000
mean         0.510733
std          3.279162
min          0.000000
25%          0.000000
50%          0.070000
75%          0.290000
max        100.000000
Name: ctr, dtype: float64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `ctr_gap` = expected CTR for a page's position tier − its actual CTR.** Bigger gap = more opportunity being left on the table at that rank.

This is a **defined-rule proxy**, not an observed future outcome — `expected_ctr` comes from a fixed benchmark curve I set per position bucket (position 1 ≈ 28%, down to position 20+ ≈ 0.3%), not from anything the data itself proved. That's a real limitation: the benchmark is an external assumption, and I'm not yet checking whether closing the gap actually raises future CTR. For the capstone I'd want to move toward an observed-outcome label instead — did CTR change over the next 30 days after review — but for this week's framing, the rule-based gap is what I have and I'm labeling it honestly as a proxy, not a discovery.

In [2]:
def expected_ctr(pos):
    if pos <= 1: return 28.0
    elif pos <= 2: return 15.0
    elif pos <= 3: return 10.0
    elif pos <= 5: return 6.0
    elif pos <= 10: return 2.5
    elif pos <= 20: return 1.0
    else: return 0.3

qualified["expected_ctr"] = qualified.avg_position.apply(expected_ctr)
qualified["ctr_gap"] = qualified["expected_ctr"] - qualified["ctr"]

print(qualified["ctr_gap"].describe())
print("pages with a positive gap (underperforming their position):", (qualified.ctr_gap > 0).sum())


count    30000.000000
mean         2.779657
std          6.503980
min        -99.700000
25%          0.300000
50%          1.000000
75%          2.500000
max         28.000000
Name: ctr_gap, dtype: float64
pages with a positive gap (underperforming their position): 28133


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@50**, checked against the transparent `low_ctr_visible_page` rule from w01 as an interim reference (impressions ≥500, position ≤20, CTR <0.5%).

Why this metric and not plain accuracy or correlation: the list is only ever consumed 50-at-a-time by one reviewer per week, so what matters is whether the *top* of my ranked list is good, not how the model does across all 30,000 rows. "Good" means: of my model's top 50 `ctr_gap` picks, most should also satisfy the independent rule-based definition of an opportunity page — that's the check that the ranking isn't just reordering noise. This is still a proxy check, not proof the model is right — real validation needs a human reviewer reading the top 50, or (better, later) a future-outcome check.

In [3]:
low_ctr_rule = (
    (qualified.impressions_90d >= 500)
    & (qualified.avg_position > 0) & (qualified.avg_position <= 20)
    & (qualified.ctr < 0.5)
)

top50 = qualified.sort_values("ctr_gap", ascending=False).head(50)
precision_at_50 = low_ctr_rule.loc[top50.index].mean()
print("precision@50 vs. rule-based baseline:", round(precision_at_50, 3))


precision@50 vs. rule-based baseline: 0.0


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = **one content item, for one client, aggregated over the fixed 90-day window in this snapshot** (`content_id` + `client_id`, grain matches `dim_content`/starter CSV — not a daily fact row). Filtered to `impressions_90d > 0` and `content_age_days >= 90`, same as w01, so every row has enough traffic and age to compute a stable CTR.

In [4]:
cols = ["content_id", "client_id", "avg_position", "impressions_90d", "clicks_90d",
        "ctr", "expected_ctr", "ctr_gap"]
print("shape:", qualified[cols].shape)
print("one row per content_id:", qualified.content_id.is_unique)
qualified[cols].sort_values("ctr_gap", ascending=False).head(5)


shape: (30000, 8)
one row per content_id: True


,content_id,client_id,avg_position,impressions_90d,clicks_90d,ctr,expected_ctr,ctr_gap
29971,content_92a5d2709aa9,client_25fc0e7096,0.0,3,0,0.0,28.0,28.0
11,content_5a3e876cf7f7,client_d4735e3a26,0.0,1,0,0.0,28.0,28.0
29992,content_23dce6a656e6,client_e29c9c180c,0.0,1,0,0.0,28.0,28.0
29995,content_c322796023c8,client_e29c9c180c,0.0,1,0,0.0,28.0,28.0
24822,content_24b1663899ca,client_e29c9c180c,0.0,1,0,0.0,28.0,28.0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Two pieces of evidence from this dataset:

1. **The rule's own threshold is brittle.** Nudging the CTR cutoff from 0.3% to 0.7% (a small, arguably-arbitrary change) swings the flagged count from 7,555 to 10,769 pages — a ~43% swing off one number I picked by hand. A single if-statement has no natural place to draw that line.
2. **Pages at the same position don't behave the same.** Restricting to just position 8-10 (2,857 pages, so position is held roughly constant), CTR still ranges from 0.0 to 100 with a std of ~2.3 — position alone doesn't explain it. `ctr_gap` shows real (if modest) correlation with `word_count` (-0.11), `days_since_last_update` (-0.12), and `scroll_rate` (+0.10) — several weak signals combining, which is exactly the kind of pattern a single if-statement can't hold but a model can weigh together.

In [5]:
# 1. threshold brittleness
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    n = len(qualified[(qualified.impressions_90d >= 500) & (qualified.avg_position > 0)
                       & (qualified.avg_position <= 20) & (qualified.ctr < t)])
    print(f"ctr threshold <{t}: {n} pages flagged")

# 2. same-position variance + multivariate correlation
tier10 = qualified[(qualified.avg_position > 8) & (qualified.avg_position <= 10)]
print("\nposition 8-10, n =", len(tier10), "| ctr std:", round(tier10.ctr.std(), 2),
      "| min/max:", tier10.ctr.min(), "/", tier10.ctr.max())

corr_cols = ["ctr_gap", "word_count", "days_since_last_update", "scroll_rate"]
print("\ncorrelation with ctr_gap:")
print(qualified[corr_cols].corr()["ctr_gap"].round(3))


ctr threshold <0.3: 7555 pages flagged
ctr threshold <0.4: 8877 pages flagged
ctr threshold <0.5: 9759 pages flagged
ctr threshold <0.6: 10364 pages flagged
ctr threshold <0.7: 10769 pages flagged

position 8-10, n = 2857 | ctr std: 2.29 | min/max: 0.0 / 100.0

correlation with ctr_gap:
ctr_gap                   1.000
word_count               -0.107
days_since_last_update   -0.118
scroll_rate               0.099
Name: ctr_gap, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.